In [0]:
WITH all_trips AS (
    SELECT * FROM yellow_tripdata_january
    UNION ALL
    SELECT * FROM yellow_tripdata_february
    UNION ALL
    SELECT * FROM yellow_tripdata_march
),

weather_flags AS (
    SELECT
        DATE                                                    AS weather_date,
        PRCP,
        CASE
            WHEN PRCP > 25 OR COALESCE(WT16, 0) = 1 THEN 1
            ELSE 0
        END                                                     AS is_rainy_day
    FROM `ny-weather-data`
),

clean_trips AS (
    SELECT
        t.tpep_pickup_datetime,
        t.PULocationID,
        t.DOLocationID,
        t.total_amount,
        t.passenger_count,
        t.fare_amount,
        DATE(t.tpep_pickup_datetime)                            AS pickup_date,
        w.is_rainy_day,
        TIMESTAMP_SECONDS(
            FLOOR(UNIX_TIMESTAMP(t.tpep_pickup_datetime) / 240) * 240
        )                                                       AS time_bucket_4min
    FROM all_trips t
    INNER JOIN weather_flags w
        ON DATE(t.tpep_pickup_datetime) = w.weather_date
    WHERE
        t.fare_amount       > 0
        AND t.total_amount  > 0
        AND t.trip_distance > 0
        AND t.passenger_count BETWEEN 1 AND 6
),

shareable_groups AS (
    SELECT
        PULocationID,
        DOLocationID,
        is_rainy_day,
        COUNT(*)                                                AS trips_in_group,
        AVG(total_amount)                                       AS avg_fare
    FROM clean_trips
    GROUP BY
        pickup_date,
        is_rainy_day,
        PULocationID,
        DOLocationID,
        time_bucket_4min
    HAVING COUNT(*) >= 2
)

SELECT
    PULocationID                                                AS pickup_zone,
    DOLocationID                                                AS dropoff_zone,
    CASE WHEN is_rainy_day = 1 THEN 'Rainy' ELSE 'Dry'
    END                                                         AS weather_type,
    COUNT(*)                                                    AS times_this_route_was_shareable,
    SUM(trips_in_group)                                         AS total_trips_on_this_route,
    ROUND(AVG(avg_fare), 2)                                     AS avg_fare_usd
FROM shareable_groups
GROUP BY PULocationID, DOLocationID, is_rainy_day
ORDER BY times_this_route_was_shareable DESC
LIMIT 15;